In [ ]:
%load_ext autoreload
%autoreload 2

This notebook demonstrates the fitting routine adapted from the powerlaw module.

In [ ]:
from wildfire_stats import config as cfg
from wildfire_stats.data import io as dio
from wildfire_stats.fitting import powerlaw as wfpl

P = cfg.load_paths()
A = cfg.load_assumptions()

In [ ]:
mtbs_gdf = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp")
data = mtbs_gdf["AREA_SQKM"].dropna().to_numpy()

This function uses MLE to fit the 9 distributions tested to the data (7 distributions, then lognormal and Weibull are fit with 2 fits: conditional and shift-fit (titled _excess)), and runs R bootstrap samples to generate an error estimate. 

In [ ]:
params = wfpl.summarize_parameters_bootstrap(
    data, R=A["static_R_boot"], xmin=A["min_fire"], random_state=A["random_seed"]
)
params

This function calculates the likelihood of each distribution fit and displays the best fitting distribution and loglikelihoods of the other fits relative to the best one.

In [ ]:
R, best = wfpl.likelihood_matrix_and_best(data, xmin=A["min_fire"])
print("Best-fitting distribution:", best)
R

In [ ]:
ax = wfpl.plot_ccdf_with_selected_fits(
    data, xmin=A["min_fire"], which=["weibull", "power_law"]
)

Verify that the fitting routine yields similar results when applied to the region in time and space where all 3 datasets overlap. The following code creates the overlapping spatial/temporal subsets.

In [ ]:
mtbs_proj = dio.load_mtbs(P.mtbs / "S_USA.MTBS_BURN_AREA_BOUNDARY.shp", area_col="area_km2")
idaho_gdf = dio.load_idaho(P.idaho / "HFD_HistoricFiresDatabase7_0.gdb")

mtbs_proj = dio.filter_years(mtbs_proj, "YEAR", A["modis_years"])
idaho_gdf = dio.filter_years(idaho_gdf, "FIRE_YEAR", A["modis_years"])

mtbs_overlap = dio.overlap_with(mtbs_proj, idaho_gdf, dedup_col="FIRE_ID")

gfa_combined_gdf = dio.load_gfa(P.gfa)
gfa_overlap = dio.overlap_with(gfa_combined_gdf, idaho_gdf, dedup_col="geometry")

In [ ]:
mtbs_data = mtbs_overlap["area_km2"].dropna().to_numpy()
gfa_data = gfa_overlap["area_km2"].dropna().to_numpy()
id_data = idaho_gdf["area_km2"].dropna().to_numpy()

R_QUICK = 10  # quick cross-dataset check (original used R=10, not static_R_boot)
mtbs_params = wfpl.summarize_parameters_bootstrap(mtbs_data, R=R_QUICK, xmin=A["min_fire"], random_state=A["random_seed"])
gfa_params = wfpl.summarize_parameters_bootstrap(gfa_data, R=R_QUICK, xmin=A["min_fire"], random_state=A["random_seed"])
id_params = wfpl.summarize_parameters_bootstrap(id_data, R=R_QUICK, xmin=A["min_fire"], random_state=A["random_seed"])

mtbs_R, mtbs_best = wfpl.likelihood_matrix_and_best(mtbs_data, xmin=A["min_fire"])
gfa_R, gfa_best = wfpl.likelihood_matrix_and_best(gfa_data, xmin=A["min_fire"])
id_R, id_best = wfpl.likelihood_matrix_and_best(id_data, xmin=A["min_fire"])

mtbs_params, gfa_params, id_params

In [ ]:
mtbs_R, gfa_R, id_R

Close relative performance of different distributions on all 3 datasets!